# Lesson 20 — Spark Fundamentals:

Bu notebook Dərs 20-nin (Slayd 12: *Hands-on Demo*) canlı hissəsi üçün nəzərdə tutulub. Məqsəd — nəzəri hissədə gördüyümüz konsepsiyaları (Driver/Executor, Job/Stage/Task, DAG, Lazy Evaluation, Narrow vs Wide Transformation) canlı Spark UI üzərində müşahidə etməkdir.

**Mühit tələbləri:**
- Spark standalone cluster işlək olmalıdır (`spark://spark-master:7077`)
- MinIO (S3A) işlək olmalıdır (`http://minio:9000`)
- Bu notebook-u icra etdikdən sonra brauzerdə **http://localhost:4040** (Spark UI) və **http://localhost:8080** (Standalone Master UI) səhifələrini paralel aç.



## 1. SparkSession qurulması 

Aşağıdakı kod standalone cluster-ə qoşulur və MinIO-ya S3A protokolu ilə əlaqə üçün konfiqurasiya edir. Bu, **client mode** nümunəsidir — Driver bu notebook kernel-ində işləyəcək (Slayd 6).


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

spark = (
    SparkSession.builder
    .appName("Lesson20-Fundamentals-Practical")
    .master("spark://spark-master:7077")

    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:/spark-events")

    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "matrix")
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

spark.version

'3.5.2'

## 2. Data yaratmaq və yazmaq (Slayd 8, 12)

1 milyon sətirlik sintetik DataFrame yaradıb MinIO-ya parquet formatında yazacağıq.


In [19]:
df = (
    spark.range(0, 1_000_000)
    .withColumn("category", (F.col("id") % 5).cast("int"))
    .withColumn("amount", (F.rand(seed=42) * 1000).cast("double"))
    .select("category", "amount")
    .repartition(8)
)

base_path = "s3a://matrix/lesson20_base_parquet"
df.write.mode("overwrite").parquet(base_path)
print("write:", base_path)

write: s3a://matrix/lesson20_base_parquet


## 3. Lazy Evaluation canlı nümunəsi (Slayd 10)

Aşağıdakı hüceyrəni icra et — **heç bir Job yaranmayacaq**, çünki hamısı transformation-dır.


In [20]:
t0 = time.time()

df2 = spark.read.parquet(base_path)          # transformation
df3 = df2.filter(F.col("amount") > 500)      # transformation
df4 = df3.withColumn("tier", F.lit("high"))  # transformation

print(f"Transformations {time.time() - t0:.4f} san")

Transformations 0.1092 san


In [21]:
t0 = time.time()

result_count = df4.count()   

print(f"count() : {result_count}")
print(f"time: {time.time() - t0:.4f} saniyə")


count() : 498620
time: 0.2086 saniyə


## 4. Narrow vs Wide Transformation 



In [23]:
# NARROW: filter —
narrow_result = df2.filter(F.col("category") == 1)
narrow_count = narrow_result.count()
# narrow_result.write.mode("overwrite").parquet("s3a://matrix/narrow_test")
print("Narrow transformation", narrow_count)

Narrow transformation 200000


In [24]:
# WIDE: groupBy + agg
wide_result = (
    df2.repartition(16, "category")
       .groupBy("category")
       .agg(F.sum("amount").alias("total_amount"))
       .orderBy(F.desc("total_amount"))
)
wide_result.show()

+--------+-------------------+
|category|       total_amount|
+--------+-------------------+
|       4|9.995416103507642E7|
|       0| 9.99448876041471E7|
|       2|9.985687980318572E7|
|       3|9.980381663076152E7|
|       1| 9.97750313774988E7|
+--------+-------------------+



In [25]:

distinct_categories = df2.select("category").distinct()
distinct_categories.show()

+--------+
|category|
+--------+
|       1|
|       3|
|       4|
|       2|
|       0|
+--------+



## 5. Physical Plan-a baxış — `explain()`



In [17]:
wide_result.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (7)
+- Sort (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- HashAggregate (3)
            +- Exchange (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [category#26, amount#27]
Batched: true
Location: InMemoryFileIndex [s3a://matrix/lesson20_base_parquet]
ReadSchema: struct<category:int,amount:double>

(2) Exchange
Input [2]: [category#26, amount#27]
Arguments: hashpartitioning(category#26, 16), REPARTITION_BY_NUM, [plan_id=473]

(3) HashAggregate
Input [2]: [category#26, amount#27]
Keys [1]: [category#26]
Functions [1]: [partial_sum(amount#27)]
Aggregate Attributes [1]: [sum#86]
Results [2]: [category#26, sum#87]

(4) HashAggregate
Input [2]: [category#26, sum#87]
Keys [1]: [category#26]
Functions [1]: [sum(amount#27)]
Aggregate Attributes [1]: [sum(amount#27)#76]
Results [2]: [category#26, sum(amount#27)#76 AS total_amount#77]

(5) Exchange
Input [2]: [category#26, total_amount#77]
Arguments: rangepartitioning(

## 6. Job/Stage/Task-ları proqramatik izləmək (Slayd 8)

Spark UI-a əlavə olaraq, `statusTracker` vasitəsilə Job/Stage məlumatlarını kod daxilində də ala bilərik.


In [27]:
tracker = spark.sparkContext.statusTracker()
job_ids = tracker.getJobIdsForGroup()

print(f" SparkSession Job sayı: {len(job_ids)}")
print(f"Job ID-ləri: {sorted(job_ids)}")

if job_ids:
    last_job = tracker.getJobInfo(max(job_ids))
    print(f"\nJob Stage ID: {last_job.stageIds}")

 SparkSession Job sayı: 32
Job ID-ləri: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]

Job Stage ID: [I@7560e79b


## 8. Sonlandırma

İşin bitəndə SparkSession-u həmişə bağla — resurslar cluster-ə geri qaytarılsın.


In [ ]:
spark.stop()